# MDDARP Results Analysis
This notebook evaluates the final objective values across all instances, methods, and seeds.
**Memory Efficient:** Only the `summary.objective` field is loaded from each JSON output file.

In [3]:
import json
import os
import pandas as pd
import numpy as np

# Define instances, seeds, and method path templates
instances = [
    "md1a", "md2a", "md3a", "md4a", "md5a", "md6a", "md7a", "md8a", "md9a", "md10a",
    "md1b", "md2b", "md3b", "md4b", "md5b", "md6b", "md7b", "md8b", "md9b", "md10b"
]
seeds = range(1, 21)

methods = {
    "Base": "../data/run_base_20/outputs/sol_{inst}_seed{seed}.json",
    "NR": "../data/run_nr/outputs/sol_{inst}_seed{seed}.json",
    "NONE": "../data/run_intra_route/outputs/sol_{inst}_seed_{seed}_NONE.json",
    "UNRESTRICTED": "../data/run_intra_route/outputs/sol_{inst}_seed_{seed}_UNRESTRICTED.json",
    "SP": "../data/run_set_based/outputs/sol_{inst}_seed_{seed}_ALNS_SP.json",
    "SC": "../data/run_set_based/outputs/sol_{inst}_seed_{seed}_ALNS_SC.json"
}

data = []

for inst in instances:
    for method, path_template in methods.items():
        for seed in seeds:
            path = path_template.format(inst=inst, seed=seed)
            if os.path.exists(path):
                try:
                    with open(path, 'r') as f:
                        content = json.load(f)
                        obj = content.get("summary", {}).get("objective", None)
                        if obj is not None:
                            data.append({
                                "Instance": inst,
                                "Method": method,
                                "Seed": seed,
                                "Objective": obj
                            })
                except Exception as e:
                    pass

df = pd.DataFrame(data)
print(f"Loaded {len(df)} experiment results successfully across {df['Instance'].nunique()} instances.")

Loaded 2400 experiment results successfully across 20 instances.


In [4]:
# 1. Best objective found per instance and method (across all 20 seeds)
best_per_method_inst = df.groupby(['Instance', 'Method'])['Objective'].min().unstack()
print("--- Best Objective per Instance & Method ---")
display(best_per_method_inst)

# 2. Absolute best objective across all methods for each instance
absolute_best = best_per_method_inst.min(axis=1)

# 3. Ratio relative to the best found (Objective / Best_Overall)
ratio_df = best_per_method_inst.divide(absolute_best, axis=0)
print("--- Ratio to Best Found per Instance & Method ---")
display(ratio_df)

--- Best Objective per Instance & Method ---


Method,Base,NONE,NR,SC,SP,UNRESTRICTED
Instance,,,,,,
md10a,1143.00313,1151.21406,1356.08072,1165.80296,1165.52942,1152.81145
md10b,1136.20323,1133.42448,1245.26277,1164.96528,1151.26179,1120.41487
md1a,308.56678,308.56678,308.56678,308.56678,308.56678,308.56678
md1b,255.98045,255.98045,256.14969,255.98045,255.98045,255.98045
md2a,530.01419,532.55052,525.59808,526.08871,526.08871,529.93187
md2b,462.52354,465.68584,474.64786,462.52354,462.52354,460.16271
md3a,683.60992,679.80937,701.04134,679.62623,679.62623,678.76619
md3b,605.85483,609.72965,625.47521,605.35055,605.35055,607.44202
md4a,825.66922,823.44022,861.11174,825.66922,825.66922,826.98499


--- Ratio to Best Found per Instance & Method ---


Method,Base,NONE,NR,SC,SP,UNRESTRICTED
Instance,,,,,,
md10a,1.000000,1.007184,1.186419,1.019947,1.019708,1.008581
md10b,1.014092,1.011611,1.111430,1.039762,1.027532,1.000000
md1a,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
md1b,1.000000,1.000000,1.000661,1.000000,1.000000,1.000000
md2a,1.008402,1.013228,1.000000,1.000933,1.000933,1.008245
md2b,1.005130,1.012003,1.031478,1.005130,1.005130,1.000000
md3a,1.007136,1.001537,1.032817,1.001267,1.001267,1.000000
md3b,1.000833,1.007234,1.033245,1.000000,1.000000,1.003455
md4a,1.002707,1.000000,1.045749,1.002707,1.002707,1.004305


In [5]:
# 4. Summary Table: Wins, Average Best Objective, and Average Ratio to Best
tolerance = 1e-5
wins_mask = np.abs(best_per_method_inst.values - absolute_best.values[:, None]) < tolerance
wins_count = pd.Series(wins_mask.sum(axis=0), index=best_per_method_inst.columns, name="Wins")

avg_ratio = ratio_df.mean().rename("Avg Ratio to Best")
avg_obj = best_per_method_inst.mean().rename("Avg Best Objective")

summary_table = pd.DataFrame({
    "Wins": wins_count,
    "Avg Best Objective": avg_obj,
    "Avg Ratio to Best": avg_ratio
})

print(
    "--- Overall Performance Summary ---"
)
display(summary_table.sort_values(by="Wins", ascending=False))

--- Overall Performance Summary ---


,Wins,Avg Best Objective,Avg Ratio to Best
Method,,,
SP,10,735.854439,1.004688
Base,8,734.189598,1.003571
SC,8,737.342518,1.006164
UNRESTRICTED,8,734.367013,1.003503
NONE,5,736.185027,1.006046
NR,3,780.188054,1.052312
